# Notebook 01: Data Cleaning

**Project:** Forecasting Religious Tourism — Machine Learning Analysis of Pilgrimage Patterns to Sacred Sites in Nepal

**Purpose:** Load the raw messy dataset, identify data quality issues, fix each one systematically, and save a clean dataset for analysis.

**Input:** `data/pilgrimage_data_raw.csv`

**Output:** `data/pilgrimage_data_clean.csv`

---

## Step 1: Import Libraries

We import pandas for data handling, numpy for numerical operations, and warnings to keep the notebook clean.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import warnings

# Suppress unnecessary warnings to keep output clean
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

Libraries imported successfully


## Step 2: Load Raw Data

We load the raw CSV file. This file contains messy, real-world-like data with various quality issues that we will fix in the steps below.

In [5]:
# Load the raw messy dataset
df = pd.read_csv('../data/pilgrimage_data_raw.csv')

# Show basic info
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

Rows: 766
Columns: ['date', 'site', 'total_visitors', 'domestic_visitors', 'indian_visitors', 'international_visitors', 'temperature_c', 'rainfall_mm', 'festival', 'season']

First 5 rows:


,date,site,total_visitors,domestic_visitors,indian_visitors,international_visitors,temperature_c,rainfall_mm,festival,season
0,2020-12,Janakpur Dham,6051,3830.0,1784.0,437.0,14.2,3.0,0,winter
1,2019-08,Janakpur Dham,13139,8195.0,3526.0,1418.0,30.9,275.0,0,monsoon
2,2018-04,Lumbini,144304,82731.0,38665.0,22908.0,27.2,24.0,0,pre-monsoon
3,2013-06,Pashupatinath,28202,20387.0,5292.0,2523.0,24.3,222.0,0,monsoon
4,2010-08,Janakpur Dham,5324,3375.0,1534.0,415.0,31.0,326.0,0,monsoon


## Step 3: Initial Data Inspection

Before cleaning, we need to understand what problems exist. We check data types, missing values, unique values in key columns, and look for anything unusual.

In [6]:
# Check data types — are numbers stored as text?
print('DATA TYPES:')
print(df.dtypes)
print()

# Check missing values per column
print('MISSING VALUES:')
print(df.isnull().sum())
print()

# Check unique site names — should be exactly 4
print('UNIQUE SITE NAMES:')
print(df['site'].unique())
print(f'Count: {df["site"].nunique()}')
print()

# Check unique season labels — should be exactly 4
print('UNIQUE SEASON LABELS:')
print(df['season'].unique())
print(f'Count: {df["season"].nunique()}')
print()

# Check date format samples
print('SAMPLE DATE VALUES:')
print(df['date'].sample(10).values)

DATA TYPES:
date                       object
site                       object
total_visitors             object
domestic_visitors         float64
indian_visitors           float64
international_visitors    float64
temperature_c             float64
rainfall_mm               float64
festival                    int64
season                     object
dtype: object

MISSING VALUES:
date                       0
site                       0
total_visitors             9
domestic_visitors          9
indian_visitors            9
international_visitors     9
temperature_c             11
rainfall_mm               10
festival                   0
season                     0
dtype: int64

UNIQUE SITE NAMES:
['Janakpur Dham' 'Lumbini' 'Pashupatinath' 'Muktinath' 'LUMBINI'
 'Lumbini Garden' ' Lumbini' 'MUKTINATH' 'PASHUPATINATH'
 'Muktinath Temple' 'Pashupatinath Temple' 'Janakpur' 'Pashupati Nath']
Count: 13

UNIQUE SEASON LABELS:
['winter' 'monsoon' 'pre-monsoon' 'post-monsoon' 'Monsoon' '  winte

### Issues Identified

From the inspection above, we can see the following problems:

1. **Inconsistent date formats** — some dates show as '2018-01', others as 'Jan-2018' or 'January 2018'
2. **Inconsistent site names** — 'Pashupatinath', 'pashupatinath', 'PASHUPATINATH', 'Pashupati Nath' etc.
3. **Duplicate rows** — same month and site appearing more than once
4. **Visitor counts as text** — numbers with commas like '12,315' stored as strings
5. **Missing values** — NaN in visitor counts, temperature, and rainfall columns
6. **Negative rainfall** — impossible values like -5 or -12
7. **Extra whitespace in season** — '  winter  ' instead of 'winter'
8. **Wrong season labels** — 'Monsoon', 'WINTER', 'rainy' instead of standardised names
9. **Outliers** — extreme values like 999999 or 9999999 in visitor counts
10. **Rows not sorted** — data is shuffled randomly

We will fix each issue one by one in the steps below.

## Step 4: Fix Inconsistent Site Names

The site column has variations like 'pashupatinath', 'PASHUPATINATH', 'Pashupati Nath', 'Pashupatinath Temple' etc. We need to map all of these to 4 standard names.

In [7]:
# First, let's see all unique site names before fixing
print('BEFORE:', df['site'].unique())
print(f'Count: {df["site"].nunique()}')
print()

# Strip whitespace from site names
df['site'] = df['site'].str.strip()

# Convert to lowercase for consistent matching
df['site_lower'] = df['site'].str.lower()

# Map all variations to standard names
def standardise_site(name):
    name = name.lower().strip()
    if 'pashupati' in name:
        return 'Pashupatinath'
    elif 'lumbini' in name:
        return 'Lumbini'
    elif 'mukti' in name or 'muktinath' in name:
        return 'Muktinath'
    elif 'janakpur' in name or 'janakpurdham' in name:
        return 'Janakpur Dham'
    else:
        return name  # Return as-is if no match (for investigation)

# Apply the function
df['site'] = df['site'].apply(standardise_site)

# Remove the helper column
df.drop('site_lower', axis=1, inplace=True)

# Verify — should now be exactly 4 unique names
print('AFTER:', df['site'].unique())
print(f'Count: {df["site"].nunique()}')

BEFORE: ['Janakpur Dham' 'Lumbini' 'Pashupatinath' 'Muktinath' 'LUMBINI'
 'Lumbini Garden' ' Lumbini' 'MUKTINATH' 'PASHUPATINATH'
 'Muktinath Temple' 'Pashupatinath Temple' 'Janakpur' 'Pashupati Nath']
Count: 13

AFTER: ['Janakpur Dham' 'Lumbini' 'Pashupatinath' 'Muktinath']
Count: 4


## Step 5: Fix Inconsistent Date Formats

Dates appear in many formats: '2018-01', 'Jan-2018', 'January 2018', '01/01/2018', '2018/01', '01-2018'. We need to convert all of them to a standard 'YYYY-MM' format.

In [8]:
# Show problematic dates before fixing
print('SAMPLE DATES BEFORE:')
print(df['date'].sample(15).values)
print()

# Function to parse various date formats into standard YYYY-MM
def standardise_date(date_str):
    date_str = str(date_str).strip()
    
    # Try multiple date formats
    formats_to_try = [
        '%Y-%m',       # 2018-01
        '%b-%Y',       # Jan-2018
        '%B %Y',       # January 2018
        '%d/%m/%Y',    # 01/01/2018
        '%Y/%m',       # 2018/01
        '%m-%Y',       # 01-2018
    ]
    
    for fmt in formats_to_try:
        try:
            parsed = pd.to_datetime(date_str, format=fmt)
            return parsed.strftime('%Y-%m')  # Return standard format
        except (ValueError, TypeError):
            continue
    
    # If none of the formats work, try pandas auto-detection
    try:
        parsed = pd.to_datetime(date_str)
        return parsed.strftime('%Y-%m')
    except:
        return None  # Mark as None for investigation

# Apply the function
df['date'] = df['date'].apply(standardise_date)

# Check for any dates that could not be parsed
failed = df['date'].isnull().sum()
print(f'Failed to parse: {failed} dates')

# Show dates after fixing
print('\nSAMPLE DATES AFTER:')
print(df['date'].sample(10).values)

SAMPLE DATES BEFORE:
['2016-10' '2015-09' '2014-09' '2022-05' '2022-02' '2019-03' '2024-06'
 '2018-12' 'July 2016' '2015-12' '2022-06' '2016-03' '2016-04' '2020-02'
 '2024-10']

Failed to parse: 0 dates

SAMPLE DATES AFTER:
['2022-07' '2019-02' '2016-12' '2025-10' '2017-06' '2024-04' '2025-06'
 '2024-08' '2016-01' '2022-03']


## Step 6: Fix Visitor Counts Stored as Text

Some visitor count values are stored as text with commas (e.g., '12,315' instead of 12315). We need to remove commas and convert to numeric.

In [9]:
# Check current data type of total_visitors
print('Data type before:', df['total_visitors'].dtype)
print()

# Show some problematic values
print('Sample values:')
print(df['total_visitors'].sample(10).values)
print()

# Remove commas from all visitor columns and convert to numeric
visitor_columns = ['total_visitors', 'domestic_visitors', 'indian_visitors', 'international_visitors']

for col in visitor_columns:
    # Convert to string first, remove commas, then convert to numeric
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')  # 'coerce' turns invalid values into NaN

print('Data type after:', df['total_visitors'].dtype)
print(f'Missing values in total_visitors: {df["total_visitors"].isnull().sum()}')

Data type before: object

Sample values:
['111342' '92245' '136,927' '149441' '80754' '649' '6961' '79772' '75214'
 '16596']

Data type after: float64
Missing values in total_visitors: 9


## Step 7: Remove Duplicate Rows

Some rows appear more than once (same date and site). We keep only the first occurrence and remove the rest.

In [10]:
# Count duplicates before removal
duplicates = df.duplicated(subset=['date', 'site'], keep='first').sum()
print(f'Duplicate rows found: {duplicates}')

# Remove duplicates — keep the first occurrence
df = df.drop_duplicates(subset=['date', 'site'], keep='first')

# Verify
print(f'Rows after removing duplicates: {len(df)}')
print(f'Remaining duplicates: {df.duplicated(subset=["date", "site"], keep="first").sum()}')

Duplicate rows found: 5
Rows after removing duplicates: 761
Remaining duplicates: 0


## Step 8: Remove Outliers and Data Entry Errors

Some rows have clearly wrong visitor counts like 999999, 9999999, 0, or negative values. These are data entry errors, not real data. We identify and remove them.

In [11]:
# Check for extreme values in total_visitors
print('VISITOR COUNT STATISTICS:')
print(df['total_visitors'].describe())
print()

# Show suspicious values (too high or zero/negative)
suspicious = df[(df['total_visitors'] > 500000) | (df['total_visitors'] <= 0)]
print(f'Suspicious rows found: {len(suspicious)}')
if len(suspicious) > 0:
    print(suspicious[['date', 'site', 'total_visitors']])
print()

# Remove rows where total_visitors is clearly an error
# Threshold: values above 500,000 for a single month are likely errors
# (Lumbini at peak gets ~130,000/month, so 500,000 is a safe upper bound)
# Also remove zero or negative values
before_count = len(df)
df = df[(df['total_visitors'] > 0) | (df['total_visitors'].isnull())]  # Keep NaN for now (handle later)
df = df[(df['total_visitors'] < 500000) | (df['total_visitors'].isnull())]

removed = before_count - len(df)
print(f'Outlier rows removed: {removed}')
print(f'Rows remaining: {len(df)}')

VISITOR COUNT STATISTICS:
count    7.530000e+02
mean     5.772805e+04
std      3.686944e+05
min     -5.000000e+02
25%      7.024000e+03
50%      2.153000e+04
75%      6.693000e+04
max      9.999999e+06
Name: total_visitors, dtype: float64

Suspicious rows found: 5
        date           site  total_visitors
80   2025-08        Lumbini        999999.0
207  2023-11  Janakpur Dham       9999999.0
226  2024-04        Lumbini          -500.0
361  2019-03  Pashupatinath          -500.0
420  2011-11        Lumbini        888888.0

Outlier rows removed: 5
Rows remaining: 756


## Step 9: Fix Negative Rainfall Values

Rainfall cannot be negative. Values like -5 or -12 are data entry errors. We replace them with NaN so they can be handled with other missing values.

In [12]:
# Convert rainfall to numeric first (in case it is stored as text)
df['rainfall_mm'] = pd.to_numeric(df['rainfall_mm'], errors='coerce')
df['temperature_c'] = pd.to_numeric(df['temperature_c'], errors='coerce')

# Check for negative rainfall
negative_rain = df[df['rainfall_mm'] < 0]
print(f'Negative rainfall values found: {len(negative_rain)}')
if len(negative_rain) > 0:
    print(negative_rain[['date', 'site', 'rainfall_mm']])

# Replace negative rainfall with NaN
df.loc[df['rainfall_mm'] < 0, 'rainfall_mm'] = np.nan

print(f'\nNegative rainfall values after fix: {(df["rainfall_mm"] < 0).sum()}')

Negative rainfall values found: 3
        date           site  rainfall_mm
235  2019-05  Janakpur Dham         -5.0
456  2017-10  Janakpur Dham        -12.0
738  2014-01  Janakpur Dham         -8.0

Negative rainfall values after fix: 0


## Step 10: Fix Season Labels and Whitespace

Season values have extra spaces ('  winter  ') and inconsistent formats ('Monsoon', 'WINTER', 'rainy'). We strip whitespace and map all variations to 4 standard labels.

In [13]:
# Show unique seasons before fixing
print('BEFORE:', df['season'].unique())
print()

# Strip whitespace first
df['season'] = df['season'].astype(str).str.strip().str.lower()

# Map all variations to standard labels
season_mapping = {
    'winter': 'winter',
    'pre-monsoon': 'pre-monsoon',
    'pre monsoon': 'pre-monsoon',
    'monsoon': 'monsoon',
    'rainy': 'monsoon',
    'post-monsoon': 'post-monsoon',
    'post monsoon': 'post-monsoon',
}

df['season'] = df['season'].map(season_mapping).fillna(df['season'])

# Verify — should be exactly 4 unique values
print('AFTER:', df['season'].unique())
print(f'Count: {df["season"].nunique()}')

BEFORE: ['winter' 'monsoon' 'pre-monsoon' 'post-monsoon' 'Monsoon' '  winter  '
 'Pre-Monsoon' '  monsoon  ' '  pre-monsoon  ' '  post-monsoon  ' 'rainy'
 'WINTER']

AFTER: ['winter' 'monsoon' 'pre-monsoon' 'post-monsoon']
Count: 4


## Step 11: Handle Missing Values

Now we deal with NaN values in visitor counts, temperature, and rainfall. For time series data, we use **forward fill** (fill with the previous month's value) followed by **backward fill** for any remaining gaps at the start. This preserves the temporal continuity better than filling with the mean.

In [14]:
# Check missing values before handling
print('MISSING VALUES BEFORE:')
print(df.isnull().sum())
print()

# Sort by site and date first (required for forward fill to work correctly)
df = df.sort_values(['site', 'date']).reset_index(drop=True)

# Forward fill within each site group, then backward fill for any remaining
columns_to_fill = ['total_visitors', 'domestic_visitors', 'indian_visitors', 
                    'international_visitors', 'temperature_c', 'rainfall_mm']

for col in columns_to_fill:
    df[col] = df.groupby('site')[col].transform(
        lambda x: x.fillna(method='ffill').fillna(method='bfill')
    )

# Verify — should be zero missing values now
print('MISSING VALUES AFTER:')
print(df.isnull().sum())

MISSING VALUES BEFORE:
date                       0
site                       0
total_visitors             8
domestic_visitors          8
indian_visitors            8
international_visitors     8
temperature_c             11
rainfall_mm               13
festival                   0
season                     0
dtype: int64

MISSING VALUES AFTER:
date                      0
site                      0
total_visitors            0
domestic_visitors         0
indian_visitors           0
international_visitors    0
temperature_c             0
rainfall_mm               0
festival                  0
season                    0
dtype: int64


## Step 12: Ensure Correct Data Types

After all the cleaning, we make sure every column has the right data type: visitor counts as integers, weather as floats, festival as integer (0 or 1).

In [15]:
# Convert visitor counts to integers
visitor_columns = ['total_visitors', 'domestic_visitors', 'indian_visitors', 'international_visitors']
for col in visitor_columns:
    df[col] = df[col].astype(int)

# Ensure temperature and rainfall are floats
df['temperature_c'] = df['temperature_c'].astype(float)
df['rainfall_mm'] = df['rainfall_mm'].astype(float)

# Ensure festival is integer (0 or 1)
df['festival'] = df['festival'].astype(int)

# Check final data types
print('FINAL DATA TYPES:')
print(df.dtypes)

FINAL DATA TYPES:
date                       object
site                       object
total_visitors              int64
domestic_visitors           int64
indian_visitors             int64
international_visitors      int64
temperature_c             float64
rainfall_mm               float64
festival                    int64
season                     object
dtype: object


## Step 13: Sort and Reset Index

The raw data was randomly shuffled. We sort it by site name first, then by date within each site, and reset the row index.

In [16]:
# Sort by site and date
df = df.sort_values(['site', 'date']).reset_index(drop=True)

# Show the first few rows of each site to verify ordering
for site in df['site'].unique():
    print(f'\n--- {site} ---')
    site_df = df[df['site'] == site]
    print(f'  Date range: {site_df["date"].min()} to {site_df["date"].max()}')
    print(f'  Rows: {len(site_df)}')


--- Janakpur Dham ---
  Date range: 2010-01 to 2025-12
  Rows: 188

--- Lumbini ---
  Date range: 2010-01 to 2025-12
  Rows: 187

--- Muktinath ---
  Date range: 2010-01 to 2025-12
  Rows: 191

--- Pashupatinath ---
  Date range: 2010-01 to 2025-12
  Rows: 190


## Step 14: Final Validation

Before saving, we run a final check to make sure everything is clean. This is our quality assurance step.

In [17]:
print('=' * 60)
print('FINAL DATA QUALITY REPORT')
print('=' * 60)
print()
print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Unique sites: {df["site"].unique()}')
print(f'Unique seasons: {df["season"].unique()}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated(subset=["date", "site"]).sum()}')
print(f'Negative rainfall: {(df["rainfall_mm"] < 0).sum()}')
print(f'Zero/negative visitors: {(df["total_visitors"] <= 0).sum()}')
print()

# Summary statistics per site
print('SUMMARY PER SITE:')
print('-' * 60)
for site in sorted(df['site'].unique()):
    site_df = df[df['site'] == site]
    print(f'\n{site}:')
    print(f'  Months: {len(site_df)}')
    print(f'  Avg monthly visitors: {site_df["total_visitors"].mean():,.0f}')
    print(f'  Min monthly visitors: {site_df["total_visitors"].min():,}')
    print(f'  Max monthly visitors: {site_df["total_visitors"].max():,}')
    print(f'  Festival months: {site_df["festival"].sum()}')

FINAL DATA QUALITY REPORT

Total rows: 756
Columns: ['date', 'site', 'total_visitors', 'domestic_visitors', 'indian_visitors', 'international_visitors', 'temperature_c', 'rainfall_mm', 'festival', 'season']
Date range: 2010-01 to 2025-12
Unique sites: ['Janakpur Dham' 'Lumbini' 'Muktinath' 'Pashupatinath']
Unique seasons: ['winter' 'pre-monsoon' 'monsoon' 'post-monsoon']
Missing values: 0
Duplicate rows: 0
Negative rainfall: 0
Zero/negative visitors: 0

SUMMARY PER SITE:
------------------------------------------------------------

Janakpur Dham:
  Months: 188
  Avg monthly visitors: 15,261
  Min monthly visitors: 1,646
  Max monthly visitors: 60,714
  Festival months: 32

Lumbini:
  Months: 187
  Avg monthly visitors: 84,617
  Min monthly visitors: 5,451
  Max monthly visitors: 204,256
  Festival months: 31

Muktinath:
  Months: 191
  Avg monthly visitors: 4,524
  Min monthly visitors: 214
  Max monthly visitors: 16,017
  Festival months: 16

Pashupatinath:
  Months: 190
  Avg monthly

In [18]:
# Preview the final cleaned dataset
print('FINAL CLEANED DATA (first 10 rows):')
df.head(10)

FINAL CLEANED DATA (first 10 rows):


,date,site,total_visitors,domestic_visitors,indian_visitors,international_visitors,temperature_c,rainfall_mm,festival,season
0,2010-01,Janakpur Dham,8037,5032,2254,751,14.0,11.0,0,winter
1,2010-02,Janakpur Dham,9547,6393,2748,406,18.1,10.0,0,winter
2,2010-03,Janakpur Dham,15966,10597,4701,668,22.3,13.0,1,pre-monsoon
3,2010-05,Janakpur Dham,9066,6016,2560,490,32.8,70.0,0,pre-monsoon
4,2010-06,Janakpur Dham,6300,3967,1836,497,32.7,191.0,0,monsoon
5,2010-07,Janakpur Dham,4789,3210,1247,332,31.1,416.0,0,monsoon
6,2010-08,Janakpur Dham,5324,3375,1534,415,31.0,326.0,0,monsoon
7,2010-09,Janakpur Dham,8415,5243,2357,815,29.7,229.0,0,post-monsoon
8,2010-10,Janakpur Dham,8980,5998,2533,449,25.7,38.0,0,post-monsoon
9,2010-11,Janakpur Dham,20559,13128,5583,1848,21.6,5.0,1,post-monsoon


## Step 15: Save Clean Data

The cleaning is complete. We save the clean dataset as a new CSV file that will be used by the next notebooks (EDA, model training, etc.).

In [20]:
# Save the cleaned dataset
df.to_csv('../data/pilgrimage_data_clean.csv', index=False)

print('Clean dataset saved successfully!')
print(f'File: data/pilgrimage_data_clean.csv')
print(f'Rows: {len(df)}')
print(f'Columns: {len(df.columns)}')

Clean dataset saved successfully!
File: data/pilgrimage_data_clean.csv
Rows: 756
Columns: 10


## Cleaning Summary

| Issue | Action Taken |
|---|---|
| Inconsistent site names | Mapped all variations to 4 standard names |
| Inconsistent date formats | Parsed all formats into YYYY-MM |
| Text with commas in visitor counts | Removed commas, converted to numeric |
| Duplicate rows | Removed, keeping first occurrence |
| Outliers (999999, 0, negative) | Removed rows with impossible values |
| Negative rainfall | Replaced with NaN, then filled |
| Extra whitespace in season | Stripped and standardised |
| Wrong season labels | Mapped to 4 standard labels |
| Missing values | Forward fill within each site group |
| Shuffled rows | Sorted by site and date |

---

**Next step:** Open `02_eda.ipynb` to explore the cleaned data visually.